In [47]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [48]:
## Load the IMDB dataset word and reverse word index
word_index = imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

In [49]:
## Load the saved model
model = load_model("simplernn_imdb.h5")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 200, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 64)               │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,292,419 (4.93 MB)

 Trainable params: 1,292,417 (4.93 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [50]:
model.get_weights()

[array([[ 0.03981688,  0.0087184 , -0.02562823, ..., -0.04449245,
          0.02047644, -0.03506121],
        [-0.015612  , -0.01615481,  0.07967132, ..., -0.01159698,
         -0.00984349,  0.06208201],
        [ 0.01930537, -0.03659759,  0.00092528, ...,  0.01250311,
         -0.01448286,  0.04214562],
        ...,
        [-0.01730075, -0.02122769,  0.00712946, ..., -0.03317226,
          0.04066668, -0.02822815],
        [ 0.00990758,  0.03592184,  0.06449636, ...,  0.04349908,
         -0.05281226, -0.05061724],
        [-0.04885807, -0.05504414, -0.03277386, ..., -0.09440493,
          0.05873328, -0.02442949]], dtype=float32),
 array([[ 0.12933925,  0.03882997,  0.00307313, ..., -0.00935844,
         -0.03224751, -0.03243237],
        [-0.19550388, -0.1882893 , -0.24255268, ..., -0.07627316,
          0.15976048,  0.1542215 ],
        [-0.14687303, -0.20946799,  0.01481935, ...,  0.07374927,
          0.01217535, -0.06371944],
        ...,
        [-0.08549568, -0.04894137, -0.0

In [51]:
## Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

## Function to preprocess user input
import re

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = text.split()

    encoded_review = []
    for word in words:
        idx = word_index.get(word)
        if idx is not None and idx < 10000:
            encoded_review.append(idx + 3)
        else:
            encoded_review.append(2)

    padded_review = sequence.pad_sequences([encoded_review], maxlen=200)
    return padded_review

In [52]:
## prediction function
def predict_sentiment(review, threshold=0.5):
    preprocessed_input = preprocess_text(review)

    prediction = model.predict(preprocessed_input)
    score = float(prediction[0][0])
    sentiment = 'Positive' if score > threshold else 'Negative'

    return sentiment, score

In [53]:
## Sample user input and prediction
example_review = "The movie was fantastic! The acting was great and the plot was thrilling."

sentiment, score = predict_sentiment(example_review)

print(f"Review: {example_review}")
print(f"Sentiment: {sentiment}")
print(f"Prediction Score: {score}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
Review: The movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.6583715081214905


In [54]:
print(predict_sentiment("I hated this movie. It was boring and terrible."))
print(predict_sentiment("I loved this movie. It was amazing and fantastic!"))
print(predict_sentiment(
"This movie was absolutely terrible. The acting was bad, the story was boring and I hated the ending."
))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
('Negative', 0.17492395639419556)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
('Positive', 0.9400802254676819)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
('Negative', 0.13933955132961273)


In [55]:
print([word_index.get(w) for w in "terrible boring hated".split()])

[391, 354, 1797]


In [56]:
print(predict_sentiment("terrible boring hated"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
('Negative', 0.1667129397392273)
